In [ ]:
# Cell 1: Install all required libraries
!pip install openai sentence-transformers nltk gensim scipy numpy rouge-score -q

import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

print("✅ All dependencies installed")

✅ All dependencies installed


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Cell 2: Imports and OpenAI API setup
import os
import json
import time
import numpy as np
from scipy.spatial.distance import cosine
from scipy.special import rel_entr
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from google.colab import userdata
import nltk
from nltk import pos_tag, word_tokenize

#API Setup
os.environ["OPENAI_API_KEY"] = userdata.get("openai_api_key")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

#Models
GENERATIVE_MODEL = "gpt-4o-mini"
TARGET_MODEL     = "gpt-4o-mini"

#Load sentence transformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✅ Using model: {GENERATIVE_MODEL}")
print(f"✅ Sentence transformer loaded")


def call_llm(system_msg: str, user_msg: str,
             model: str = None,
             temperature: float = 0.7,
             max_retries: int = 6) -> str:
    if model is None:
        model = GENERATIVE_MODEL

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=temperature,
                max_tokens=800
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f"  ⚠️  Attempt {attempt+1} failed: {e}")
            print(f"  ⏳ Waiting {wait}s before retry...")
            time.sleep(wait)

    return ""


#Quick sanity check
test = call_llm("You are a helpful assistant.", "Say hello in one word.")
print(f"✅ OpenAI API working. Response: {test}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Using model: gpt-4o-mini
✅ Sentence transformer loaded
✅ OpenAI API working. Response: Hello!


In [ ]:
# Cell 3: Prompt dataset D
# Each entry: category, prompt pi, user input xi
# These are open-source style prompts (no copyright issues)

DATASET = {

    "email": [
        {"prompt": "Write a professional follow-up email after a job interview. Be concise, thankful, and restate interest in the [role].",
         "input": "Software Engineer"},
        {"prompt": "Write a cold outreach email to a [company] asking about internship openings. Keep it short and enthusiastic.",
         "input": "Google"},
        {"prompt": "Draft a polite email to reschedule a meeting with [person] due to a conflict.",
         "input": "my manager"},
        {"prompt": "Write a formal complaint email about a delayed [product] order. Be firm but professional.",
         "input": "laptop"},
        {"prompt": "Write a thank-you email to a [role] who gave a guest lecture. Mention one key takeaway.",
         "input": "professor"},
        {"prompt": "Draft an introduction email to a new [team] you are joining. Be friendly and professional.",
         "input": "engineering team"},
        {"prompt": "Write an email requesting a [day] off for personal reasons. Keep it brief and professional.",
         "input": "Friday"},
        {"prompt": "Compose an email inviting colleagues to a [event] next week. Include date and agenda briefly.",
         "input": "team lunch"},
        {"prompt": "Write a reminder email to [person] about an upcoming project deadline. Be polite but urgent.",
         "input": "the client"},
        {"prompt": "Draft an email apologizing for missing a [meeting] and proposing a reschedule.",
         "input": "budget review meeting"},
    ],

    "ads": [
        {"prompt": "Write a catchy social media advertisement for [product]. Use emojis and a call to action.",
         "input": "wireless earbuds"},
        {"prompt": "Create a persuasive ad copy for a [product] targeting young professionals. Highlight value.",
         "input": "standing desk"},
        {"prompt": "Write a short Google Ads headline and description for [product]. Focus on benefits.",
         "input": "online coding bootcamp"},
        {"prompt": "Create a Facebook ad for [product] targeting parents. Use emotional appeal.",
         "input": "children's learning app"},
        {"prompt": "Write an Instagram caption advertisement for [product]. Make it trendy and engaging.",
         "input": "skincare serum"},
        {"prompt": "Draft a 30-second radio ad script for [product]. Be energetic and memorable.",
         "input": "local pizza restaurant"},
        {"prompt": "Write a promotional ad for a [product] sale event. Include urgency and discount mention.",
         "input": "Black Friday shoe sale"},
        {"prompt": "Create a LinkedIn ad copy for a [product] aimed at HR professionals.",
         "input": "recruitment software"},
        {"prompt": "Write a billboard-style slogan and supporting copy for [product].",
         "input": "electric car"},
        {"prompt": "Draft a product launch announcement ad for [product]. Build excitement and curiosity.",
         "input": "AI writing assistant"},
    ],

    "code": [
        {"prompt": "Write a Python function that [task]. Include docstring and type hints.",
         "input": "reverses a string"},
        {"prompt": "Create a Python class for a [object] with relevant attributes and methods.",
         "input": "bank account"},
        {"prompt": "Write a SQL query to [task] from a database table called users.",
         "input": "find all users who signed up in the last 30 days"},
        {"prompt": "Write a JavaScript function that [task]. Use modern ES6+ syntax.",
         "input": "debounces a search input"},
        {"prompt": "Write a Python script to [task] using the requests library.",
         "input": "fetch data from a REST API and save it as JSON"},
        {"prompt": "Create a React component that [task]. Use hooks appropriately.",
         "input": "displays a paginated list of items"},
        {"prompt": "Write a Bash script to [task]. Add comments explaining each step.",
         "input": "back up a directory to a zip file"},
        {"prompt": "Write a Python function to [task]. Handle edge cases and add unit tests.",
         "input": "validate an email address"},
        {"prompt": "Write a CSS snippet to [task]. Make it responsive.",
         "input": "create a centered card with shadow"},
        {"prompt": "Write a Python decorator that [task]. Show an example usage.",
         "input": "measures execution time of a function"},
    ]
}

print(f"✅ Dataset loaded: {sum(len(v) for v in DATASET.values())} prompts across {len(DATASET)} categories")
for cat, prompts in DATASET.items():
    print(f"   {cat}: {len(prompts)} prompts")

✅ Dataset loaded: 30 prompts across 3 categories
   email: 10 prompts
   ads: 10 prompts
   code: 10 prompts


In [ ]:
# Cell 4: Verify call_llm is working correctly
test_response = call_llm(
    system_msg="You are a helpful assistant.",
    user_msg="What is 2 + 2? Answer in one word.",
    temperature=0.0
)
print(f"✅ call_llm working. Response: {test_response}")

✅ call_llm working. Response: Four.


In [ ]:
# Cell 5: PRSA Phase 1 — Prompt Attention Algorithm
# Implements Algorithm 1 from the paper

# The 10 linguistic dimensions used for difference analysis (Section 2.2 of paper)
LINGUISTIC_FACTORS = [
    "Characteristic", "Topic", "Argument", "Structure",
    "Style", "Tone", "Purpose", "Sentence Type", "Audience", "Background"
]

ATTENTION_THRESHOLD = 1.0   # θa from paper: factors with loss > threshold are kept
N_ITERATIONS        = 2


def generate_basic_stolen_prompt(xi: str, yi: str, attention: dict = None) -> str:
    if not attention:
        user_msg = (
            f"Based on the provided user input and output, generate their prompt.\n\n"
            f"User Input: {xi}\n"
            f"Output: {yi}\n\n"
            f"Generate the instruction/prompt that produced this output. "
            f"The prompt should NOT be specific to the user input."
        )
    else:
        attention_str = "\n".join(
            [f"- {factor}: {desc}" for factor, desc in attention.items()]
        )
        user_msg = (
            f"Based on the provided user input and output, generate their prompt.\n\n"
            f"User Input: {xi}\n"
            f"Output: {yi}\n\n"
            f"The instruction/prompt should focus on these specific characteristics "
            f"observed in the output:\n{attention_str}\n\n"
            f"Generate the instruction/prompt. It should NOT be specific to the user input."
        )

    system_msg = "You are an expert at reverse-engineering LLM instructions from outputs."
    return call_llm(system_msg, user_msg, temperature=0.0)


def analyze_output_differences(yi: str, ysi: str) -> dict:
    factors_str = ", ".join(LINGUISTIC_FACTORS)
    user_msg = (
        f"Score the similarity between Output A and Output B for each linguistic factor "
        f"on a scale of 1-10 (10=identical, 1=completely different).\n\n"
        f"Output A (Target): {yi}\n\n"
        f"Output B (Generated): {ysi}\n\n"
        f"Rate EACH of these factors: {factors_str}\n\n"
        f"Respond ONLY as valid JSON, example:\n"
        f'{{"Characteristic": 7, "Topic": 5, "Argument": 8, ...}}'
    )
    system_msg = "You are a linguistic analysis expert. Respond only with valid JSON."
    raw = call_llm(system_msg, user_msg, temperature=0.0)

    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        scores = json.loads(clean)
        losses = {k: round(10 - float(v), 2) for k, v in scores.items()
                  if k in LINGUISTIC_FACTORS}
        return losses
    except Exception as e:
        print(f"  ⚠️  JSON parse error: {e}\n  Raw: {raw[:200]}")
        return {}


def get_factor_description(yi: str, factor: str) -> str:
    user_msg = f"What is the {factor} of the following output in one sentence?\n\nOutput: {yi}"
    system_msg = "You are a linguistic expert. Answer in exactly one short sentence."
    return call_llm(system_msg, user_msg, temperature=0.0)


def run_prompt_attention_algorithm(category: str, dataset: list) -> dict:
    print(f"\n{'='*60}")
    print(f"  Running Prompt Attention Algorithm | Category: {category}")
    print(f"{'='*60}")

    attention = {}

    for iteration in range(N_ITERATIONS):
        print(f"\n  ── Iteration {iteration + 1}/{N_ITERATIONS} ──")
        factor_loss_accumulator = {}

        for idx, sample in enumerate(dataset):
            pi  = sample["prompt"]
            xi  = sample["input"]

            yi  = call_llm(pi, xi, model=TARGET_MODEL, temperature=0.7)
            psi = generate_basic_stolen_prompt(xi, yi, attention if attention else None)
            ysi = call_llm(psi, xi, model=TARGET_MODEL, temperature=0.7)
            losses = analyze_output_differences(yi, ysi)

            for factor, loss in losses.items():
                factor_loss_accumulator[factor] = \
                    factor_loss_accumulator.get(factor, 0) + loss

            print(f"    Sample {idx+1}/{len(dataset)} processed ✓")
            time.sleep(1)

        print(f"\n  Factor losses this iteration:")
        for factor, total_loss in sorted(factor_loss_accumulator.items(),
                                          key=lambda x: -x[1]):
            avg_loss = total_loss / len(dataset)
            print(f"    {factor:20s}: avg_loss = {avg_loss:.2f}", end="")

            if avg_loss > ATTENTION_THRESHOLD:
                desc = get_factor_description(yi, factor)
                attention[factor] = desc
                print(f"  ← ADDED TO ATTENTION ✓")
            else:
                print()

    print(f"\n  ✅ Final attention a* for '{category}': {list(attention.keys())}")
    return attention


#3 categories
attention_per_category = {}
for category, prompts in DATASET.items():
    attention_per_category[category] = run_prompt_attention_algorithm(category, prompts)
    time.sleep(3)

print("\n\n✅ Phase 1 Complete! Attention per category:")
for cat, att in attention_per_category.items():
    print(f"  {cat}: {list(att.keys())}")


  Running Prompt Attention Algorithm | Category: email

  ── Iteration 1/2 ──
    Sample 1/10 processed ✓
    Sample 2/10 processed ✓
    Sample 3/10 processed ✓
    Sample 4/10 processed ✓
    Sample 5/10 processed ✓
    Sample 6/10 processed ✓
    Sample 7/10 processed ✓
    Sample 8/10 processed ✓
    Sample 9/10 processed ✓
    Sample 10/10 processed ✓

  Factor losses this iteration:
    Background          : avg_loss = 3.20  ← ADDED TO ATTENTION ✓
    Style               : avg_loss = 3.10  ← ADDED TO ATTENTION ✓
    Argument            : avg_loss = 2.90  ← ADDED TO ATTENTION ✓
    Sentence Type       : avg_loss = 2.40  ← ADDED TO ATTENTION ✓
    Characteristic      : avg_loss = 2.30  ← ADDED TO ATTENTION ✓
    Structure           : avg_loss = 1.90  ← ADDED TO ATTENTION ✓
    Topic               : avg_loss = 1.20  ← ADDED TO ATTENTION ✓
    Tone                : avg_loss = 1.10  ← ADDED TO ATTENTION ✓
    Audience            : avg_loss = 0.80
    Purpose             : avg_loss = 

In [ ]:
# Cell 6: Generate the final stolen prompt ps using optimized attention a*
# This is Equation 5 from the paper: ps = G(xt, yt, a*)

def generate_final_stolen_prompt(xt: str, yt: str, attention: dict) -> str:
    """
    Generates the final stolen prompt using the optimized attention a*.
    Equation 5: ps = G(xt, yt, a*)
    """
    if not attention:
        return generate_basic_stolen_prompt(xt, yt)

    # First: extract characteristics from yt for each attention factor
    characteristics = {}
    for factor in attention:
        desc = get_factor_description(yt, factor)
        characteristics[factor] = desc

    # Build instruction characteristics string
    char_str = "\n".join([f"- {f}: {d}" for f, d in characteristics.items()])

    # Generate stolen prompt with characteristics as guidance
    user_msg = (
        f"Your task is to generate an instruction based on the provided "
        f"user input and output.\n\n"
        f"User Input: {xt}\n"
        f"Output: {yt}\n\n"
        f"The instruction should focus on the specified instruction characteristics:\n"
        f"{char_str}\n\n"
        f"Generate ONLY the instruction/prompt. "
        f"Do NOT include the user input content in the prompt itself."
    )
    system_msg = "You are an expert at reverse-engineering LLM prompts from outputs."
    return call_llm(system_msg, user_msg, temperature=0.0)


# Test on one example per category
print("Testing final stolen prompt generation...\n")
test_targets = {
    "email": {"input": "project manager", "prompt": "Write a professional networking email to connect with a [role] on LinkedIn. Mention a shared interest."},
    "ads":   {"input": "smartwatch",      "prompt": "Write a compelling product ad for [product] targeting fitness enthusiasts. Use power words."},
    "code":  {"input": "sorts a list of dictionaries by a key", "prompt": "Write a clean Python function that [task]. Include example usage."},
}

stolen_prompts_test = {}
for cat, target in test_targets.items():
    print(f"Category: {cat}")
    # Simulate yt from target prompt
    yt = call_llm(target["prompt"], target["input"], model=TARGET_MODEL, temperature=0.7)
    ps = generate_final_stolen_prompt(target["input"], yt, attention_per_category[cat])
    stolen_prompts_test[cat] = {"ps": ps, "xt": target["input"], "yt": yt}
    print(f"  Stolen Prompt: {ps[:150]}...")
    print()

Testing final stolen prompt generation...

Category: email
  Stolen Prompt: Create a professional LinkedIn connection request template that emphasizes shared interests in project management. The message should include a greeti...

Category: ads
  Stolen Prompt: Create a persuasive promotional advertisement for a high-tech smartwatch aimed at fitness enthusiasts. Highlight its key features and benefits, such a...

Category: code
  Stolen Prompt: Generate a Python function that sorts a list of dictionaries by a specified key. The function should take two parameters: the list of dictionaries and...



In [ ]:
# Cell 7: PRSA Phase 2 — Prompt Pruning
# Implements Algorithm 2 from the paper

import re
from gensim.models import KeyedVectors
import gensim.downloader as gensim_api

# Load pretrained word2vec (Google News)
print("⏳ Loading word2vec model (Google News, 300d)... this takes ~3 minutes")
w2v_model = gensim_api.load("word2vec-google-news-300")
print("✅ word2vec loaded")


def get_nouns_from_input(xt: str) -> list:
    """
    Extracts nouns from user input xt via POS tagging.
    Equation 7 from paper: K = {wi | POS(wi) ∈ 'NOUN'}
    """
    tokens = word_tokenize(xt.lower())
    tagged = pos_tag(tokens)
    nouns  = [word for word, tag in tagged if tag.startswith("NN")]
    return nouns


def compute_semantic_similarity_word2vec(word1: str, word2: str) -> float:
    """
    Cosine similarity between two words using word2vec.
    Used in Equation 8 from paper.
    """
    try:
        return w2v_model.similarity(word1.lower(), word2.lower())
    except KeyError:
        return 0.0  # word not in vocabulary


def get_candidate_related_words(ps: str, xt: str,
                                 gamma: float = 0.4) -> list:
    """
    Identifies candidate words in ps semantically similar to nouns in xt.
    Equation 8: R = {wpj | cosine(vpj, vki) >= gamma}
    Returns list of (word, max_similarity) sorted by similarity descending.
    """
    nouns_in_input = get_nouns_from_input(xt)
    if not nouns_in_input:
        print("  ⚠️  No nouns found in input, skipping pruning.")
        return []

    # Tokenize prompt words (keep only alphabetic words)
    prompt_words = [w for w in word_tokenize(ps) if w.isalpha() and len(w) > 2]

    candidates = []
    for word in prompt_words:
        max_sim = max(
            compute_semantic_similarity_word2vec(word, noun)
            for noun in nouns_in_input
        )
        if max_sim >= gamma:
            candidates.append((word, max_sim))

    # Sort by similarity descending (highest similarity = most input-related)
    candidates.sort(key=lambda x: -x[1])
    return candidates


def evaluate_masked_prompt(ps: str, words_to_mask: list,
                            xt: str, yt: str) -> float:
    """
    Masks words in ps, generates output, computes semantic similarity to yt.
    This is the evaluation function fn in Algorithm 2.
    """
    masked_ps = ps
    for word in words_to_mask:
        # Replace the word with placeholder {}
        masked_ps = re.sub(r'\b' + re.escape(word) + r'\b', '{}', masked_ps)

    # Generate output using masked prompt
    try:
        ys = call_llm(masked_ps, xt, model=TARGET_MODEL, temperature=0.7)
    except Exception:
        return 0.0

    # Compute semantic similarity between ys and yt
    emb_ys = embedder.encode(ys,  convert_to_numpy=True)
    emb_yt = embedder.encode(yt,  convert_to_numpy=True)
    sim    = 1 - cosine(emb_ys, emb_yt)
    return float(sim)


def selective_beam_search(ps: str, candidates: list,
                           xt: str, yt: str,
                           alpha: float = 0.6,
                           beam_size: int = 2,
                           eval_freq: int = 5) -> list:
    """
    Algorithm 2: Selective beam search to find best words to mask.
    Returns the best list of words to remove from ps.

    alpha     = truncation factor (how much of candidates list to search)
    beam_size = b (top beams to keep)
    eval_freq = e (evaluation frequency)
    """
    words    = [w for w, _ in candidates]
    S        = min(int(alpha * len(words)), len(words))   # line 2
    s_step   = max(1, S // eval_freq)                     # line 3
    beams    = []

    print(f"    Beam search: {len(words)} candidates, S={S}, step={s_step}")

    for i in range(1, S + 1, s_step):                     # line 4
        current_beam = words[:i]                           # line 5
        score = evaluate_masked_prompt(ps, current_beam, xt, yt)  # line 6
        beams.append((score, current_beam[:]))             # line 7
        beams.sort(key=lambda x: -x[0])                   # line 8
        beams = beams[:beam_size]                          # line 9
        print(f"      i={i}: score={score:.3f}, masking={current_beam[:3]}...")

    if not beams:
        return []
    return beams[0][1]   # line 11: return best beam


def prune_stolen_prompt(ps: str, xt: str, yt: str) -> str:
    """
    Full Phase 2 pipeline: prune ps to remove input-specific words.
    Returns the final stolen prompt with {} placeholders.
    """
    print(f"  Pruning prompt for input: '{xt}'")

    # Step 1: Get candidate related words
    candidates = get_candidate_related_words(ps, xt, gamma=0.4)
    if not candidates:
        return ps  # nothing to prune

    print(f"  Found {len(candidates)} candidate words: "
          f"{[w for w,_ in candidates[:5]]}...")

    # Step 2: Selective beam search
    words_to_mask = selective_beam_search(ps, candidates, xt, yt)

    # Step 3: Apply masking
    final_ps = ps
    for word in words_to_mask:
        final_ps = re.sub(r'\b' + re.escape(word) + r'\b', '{}', final_ps)

    print(f"  ✅ Masked words: {words_to_mask}")
    return final_ps


# Apply pruning to our test stolen prompts
print("Phase 2: Prompt Pruning\n")
final_stolen_prompts = {}

for cat, data in stolen_prompts_test.items():
    print(f"\n{'─'*50}")
    print(f"Category: {cat}")
    final_ps = prune_stolen_prompt(data["ps"], data["xt"], data["yt"])
    final_stolen_prompts[cat] = {
        "original_ps": data["ps"],
        "final_ps":    final_ps,
        "xt":          data["xt"],
        "yt":          data["yt"]
    }
    print(f"  Original: {data['ps'][:100]}...")
    print(f"  Final:    {final_ps[:100]}...")

⏳ Loading word2vec model (Google News, 300d)... this takes ~3 minutes
[==================================================] 100.0% 1662.8/1662.8MB downloaded
✅ word2vec loaded
Phase 2: Prompt Pruning


──────────────────────────────────────────────────
Category: email
  Pruning prompt for input: 'project manager'
  Found 2 candidate words: ['project', 'management']...
    Beam search: 2 candidates, S=1, step=1
      i=1: score=0.859, masking=['project']...
  ✅ Masked words: ['project']
  Original: Create a professional LinkedIn connection request template that emphasizes shared interests in proje...
  Final:    Create a professional LinkedIn connection request template that emphasizes shared interests in {} ma...

──────────────────────────────────────────────────
Category: ads
  Pruning prompt for input: 'smartwatch'
  Original: Create a persuasive promotional advertisement for a high-tech smartwatch aimed at fitness enthusiast...
  Final:    Create a persuasive promotional advertiseme

In [ ]:
# Cell 8: Evaluation — computes semantic, syntactic, structural similarity + ASR

from scipy.spatial.distance import jensenshannon


def compute_semantic_similarity(text1: str, text2: str) -> float:
    """Cosine similarity between sentence embeddings."""
    e1  = embedder.encode(text1, convert_to_numpy=True)
    e2  = embedder.encode(text2, convert_to_numpy=True)
    return float(1 - cosine(e1, e2))


def compute_structural_similarity(text1: str, text2: str) -> float:
    """
    1 / JS-divergence between character n-gram distributions.
    Higher = more similar structure.
    From paper Section 5.4 — structural similarity.
    """
    def ngram_dist(text: str, n: int = 3) -> np.ndarray:
        tokens  = list(text.lower())
        ngrams  = ["".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        vocab   = sorted(set(ngrams))
        counts  = np.array([ngrams.count(v) for v in vocab], dtype=float)
        return counts / counts.sum() if counts.sum() > 0 else counts

    d1 = ngram_dist(text1)
    d2 = ngram_dist(text2)

    # Align distributions to same vocabulary
    all_ngrams = sorted(set(list(text1.lower()) + list(text2.lower())))
    def safe_dist(text):
        tokens = list(text.lower())
        ngrams = ["".join(tokens[i:i+3]) for i in range(len(tokens)-2)]
        total  = len(ngrams) if ngrams else 1
        return np.array([ngrams.count(g) / total for g in all_ngrams[:200]])

    p = safe_dist(text1) + 1e-10
    q = safe_dist(text2) + 1e-10
    p /= p.sum()
    q /= q.sum()

    js = jensenshannon(p, q)
    return float(1 / (js + 1e-6))  # reciprocal so higher = more similar


def evaluate_attack(ps: str, pt: str, xt: str, n_test_inputs: int = 2) -> dict:
    """
    Evaluates the stolen prompt ps against the target prompt pt.
    Uses n_test_inputs new inputs to test generality.
    Returns semantic, syntactic (approx), structural similarity scores.
    """
    # Generate multiple test inputs similar to xt
    test_input_prompt = f"Give {n_test_inputs} different example inputs for this task: '{xt}'. One per line, no numbers."
    test_inputs_raw   = call_llm("Generate diverse test inputs.", test_input_prompt, temperature=0.9)
    test_inputs       = [line.strip() for line in test_inputs_raw.split("\n")
                         if line.strip()][:n_test_inputs]
    if not test_inputs:
        test_inputs = [xt]

    sem_scores  = []
    str_scores  = []

    for xi_test in test_inputs:
        # Output from target prompt (ground truth)
        yt_new  = call_llm(pt,  xi_test, model=TARGET_MODEL, temperature=0.7)
        # Output from stolen prompt
        yps_new = call_llm(ps,  xi_test, model=TARGET_MODEL, temperature=0.7)

        sem  = compute_semantic_similarity(yt_new, yps_new)
        strc = compute_structural_similarity(yt_new, yps_new)

        sem_scores.append(sem)
        str_scores.append(strc)
        time.sleep(0.3)

    results = {
        "semantic_similarity":   round(np.mean(sem_scores), 3),
        "structural_similarity": round(min(np.mean(str_scores) / 100, 1.0), 3),
    }
    return results


# ASR thresholds from paper (Section 6.4)
GAMMA_SEM = 0.75
GAMMA_STR = 0.90

print("="*60)
print("  EVALUATION RESULTS")
print("="*60)

total_attacks  = 0
successful_attacks = 0
all_results    = {}

for cat, data in final_stolen_prompts.items():
    print(f"\nCategory: {cat.upper()}")

    # Use the original target prompt for evaluation
    original_prompt = test_targets[cat]["prompt"]

    scores = evaluate_attack(
        ps=data["final_ps"],
        pt=original_prompt,
        xt=data["xt"],
        n_test_inputs=2
    )
    all_results[cat] = scores

    attack_success = (
        scores["semantic_similarity"]   >= GAMMA_SEM and
        scores["structural_similarity"] >= GAMMA_STR
    )

    total_attacks += 1
    if attack_success:
        successful_attacks += 1

    print(f"  Semantic Similarity:   {scores['semantic_similarity']:.3f}  "
          f"(threshold: {GAMMA_SEM}) {'✅' if scores['semantic_similarity'] >= GAMMA_SEM else '❌'}")
    print(f"  Structural Similarity: {scores['structural_similarity']:.3f}  "
          f"(threshold: {GAMMA_STR}) {'✅' if scores['structural_similarity'] >= GAMMA_STR else '❌'}")
    print(f"  Attack Success:        {'✅ YES' if attack_success else '❌ NO'}")

print(f"\n{'='*60}")
print(f"  ATTACK SUCCESS RATE (ASR): {successful_attacks}/{total_attacks} = "
      f"{successful_attacks/total_attacks*100:.1f}%")
print(f"{'='*60}")

  EVALUATION RESULTS

Category: EMAIL
  Semantic Similarity:   0.911  (threshold: 0.75) ✅
  Structural Similarity: 1.000  (threshold: 0.9) ✅
  Attack Success:        ✅ YES

Category: ADS
  Semantic Similarity:   0.859  (threshold: 0.75) ✅
  Structural Similarity: 1.000  (threshold: 0.9) ✅
  Attack Success:        ✅ YES

Category: CODE
  Semantic Similarity:   0.618  (threshold: 0.75) ❌
  Structural Similarity: 1.000  (threshold: 0.9) ✅
  Attack Success:        ❌ NO

  ATTACK SUCCESS RATE (ASR): 2/3 = 66.7%


In [ ]:
# Cell 9: Clean summary of everything

print("\n" + "="*70)
print("  PRSA FINAL SUMMARY")
print("="*70)

for cat in final_stolen_prompts:
    print(f"\n{'─'*70}")
    print(f"  CATEGORY: {cat.upper()}")
    print(f"{'─'*70}")
    print(f"  Target Input  : {final_stolen_prompts[cat]['xt']}")
    print(f"\n  Original Stolen Prompt (before pruning):")
    print(f"    {final_stolen_prompts[cat]['original_ps']}")
    print(f"\n  Final Stolen Prompt (after pruning):")
    print(f"    {final_stolen_prompts[cat]['final_ps']}")
    print(f"\n  Evaluation Scores:")
    print(f"    Semantic:    {all_results[cat]['semantic_similarity']}")
    print(f"    Structural:  {all_results[cat]['structural_similarity']}")

print(f"\n{'='*70}")
print(f"  Overall ASR: {successful_attacks/total_attacks*100:.1f}%")
print(f"  Paper baseline (PRSA on GPT-3.5): ~31.7%")
print(f"  Note: Our scope is smaller (3 categories vs 18 in paper)")
print(f"{'='*70}")


  PRSA FINAL SUMMARY

──────────────────────────────────────────────────────────────────────
  CATEGORY: EMAIL
──────────────────────────────────────────────────────────────────────
  Target Input  : project manager

  Original Stolen Prompt (before pruning):
    Create a professional LinkedIn connection request template that emphasizes shared interests in project management. The message should include a greeting, an introduction of the sender, a mention of a specific shared interest, a request to connect based on mutual benefits, and a closing signature. Ensure the style is courteous and suitable for networking, maintaining a formal business communication tone throughout.

  Final Stolen Prompt (after pruning):
    Create a professional LinkedIn connection request template that emphasizes shared interests in {} management. The message should include a greeting, an introduction of the sender, a mention of a specific shared interest, a request to connect based on mutual benefits, and a